# 06 — Batching and throughput

**What you will learn**

- The four batch routes, and the two properties that make them fast.
- **Why batching helps even though the GPU is already saturated** — the central
  idea here, and the one most people get backwards.
- Why raising `MAX_CONCURRENT_INFERENCES` does *not* help, with measurements.
- The two independent bounds on a batch, and how to discover the live values
  rather than assuming them.
- A measured batched-vs-sequential comparison you run on your own deployment.
- Two optimizations that were measured and did **not** work, so you do not
  spend a week on them.

**What it assumes you already did**

Notebooks [01](01-getting-started.ipynb) through
[05](05-relations.ipynb). You need the `architecture` check from 01, the
multi-task schema from 03, the inference options from 04, and relations from 05.

**Roughly how long**

About 30 minutes, including a benchmark that takes a couple of minutes to run.

## Setup

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


In [2]:
health = get("/health")
ARCH = health["architecture"]
IS_BOUNDARY = ARCH == "boundary"
print("architecture:", ARCH, "| boundary-only routes available:", IS_BOUNDARY)

architecture: boundary | boundary-only routes available: True


## First, the shape of the problem

Before any batching, look at what naive concurrency does. This is the experiment
almost everyone runs first when a bulk job is slow: send the requests from a
thread pool and hope.

Two measurements below over the same 16 requests:

- **Sequential HTTP** — one request after another. What a naive bulk client does.
- **Concurrent HTTP** — 16 requests from a thread pool.

Predict the result before running it.

In [3]:
from concurrent.futures import ThreadPoolExecutor

DOC = "Apple CEO Tim Cook announced record revenue in Cupertino."
LABELS = ["company", "person", "location"]
N = 16
payload = {"text": DOC, "labels": LABELS}

# Warm up. With MODEL_PRELOAD=0 the first request pays the model load cost and
# would otherwise dominate whichever measurement happened to go first.
post("/extract_entities", payload)
print("warm-up done | slots:", get("/health")["max_concurrent_inferences"])

warm-up done | slots: 1


In [4]:
t0 = time.perf_counter()
for _ in range(N):
    post("/extract_entities", payload)
seq_total_ms = (time.perf_counter() - t0) * 1000

print(f"sequential : {seq_total_ms:8.1f} ms total   {seq_total_ms / N:6.1f} ms/doc")

sequential :   6828.9 ms total    426.8 ms/doc


In [5]:
def one_call(_):
    r = session.post(f"{BASE_URL}/extract_entities", json=payload, timeout=TIMEOUT)
    return r.status_code


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    statuses = list(pool.map(one_call, range(N)))
conc_total_ms = (time.perf_counter() - t0) * 1000

ok = sum(1 for s in statuses if s == 200)
busy = sum(1 for s in statuses if s == 503)

print(f"concurrent : {conc_total_ms:8.1f} ms total   {conc_total_ms / N:6.1f} ms/doc")
print(f"             {ok} x 200, {busy} x 503 (busy)")
print(f"\nspeedup vs sequential: {seq_total_ms / conc_total_ms:.2f}x")

if busy:
    print("\n503s are expected: one inference slot, and anything that cannot acquire it "
          "within INFERENCE_ACQUIRE_TIMEOUT_SECONDS is rejected rather than queued forever.")

concurrent :   2135.9 ms total    133.5 ms/doc
             16 x 200, 0 x 503 (busy)

speedup vs sequential: 3.20x


### Why concurrency did not do what you hoped

The first thing to say about the numbers you just got is that **they are
unstable.** This is a shared box, and the same two cells run three times in a
row on `jarvita-agx` gave speedups of 2.76x, 1.27x and 0.78x — the last one
meaning concurrency was *slower* than going one at a time. Whatever your run
printed, do not build a plan on it.

That instability is itself the finding. Here is why the ceiling is so low and so
noisy.

With `MAX_CONCURRENT_INFERENCES=1` the server **serializes at the semaphore**.
Firing 16 requests at once does not create 16 parallel inferences; it creates one
inference and 15 requests waiting on a lock. The model work — the part that
actually costs — is exactly as serial as it was before.

What concurrency *can* overlap is the non-GPU part of a request: HTTP parsing,
validation, schema construction, JSON serialization, and network round-trip
time. On a quiet box that overhead is small next to the forward pass, so you see
1.2x and change. On a loaded box, where each request spends longer waiting
around, there is more dead time to hide and the ratio looks better — which is
why the flattering numbers show up precisely when the box is *worse*. And with
eight threads competing over one slot, scheduling noise can push it below 1.0x
outright.

So the honest summary: concurrency buys you a small, unreliable win on request
overhead and nothing at all on model throughput. It never approaches the worker
count, because there is nothing for the extra workers to do.

Past a point it actively hurts. Requests that cannot acquire the slot within
`INFERENCE_ACQUIRE_TIMEOUT_SECONDS` (default 10) come back `503`. You did not go
faster; you converted some of your work into errors you now have to retry. Your
run above may show zero 503s — a longer queue or a slower moment produces them.

This is the framing for the rest of the notebook. **The GPU is already saturated
by one request at a time.** A single Orin doing a forward pass is using the
device; there is no idle capacity for a second request to soak up. So the
question is not "how do I run more inferences at once" — it is "how do I do less
work per document".

### Raising `MAX_CONCURRENT_INFERENCES` does not fix it

The natural next thought is to raise the concurrency limit. It was measured on
`jarvita-agx`, and it makes throughput **worse**:

| `MAX_CONCURRENT_INFERENCES` | Throughput |
|---|---|
| 1 | 8.0 docs/s |
| 8 | 3.4 docs/s |

Going from one slot to eight cut throughput by more than half. Single run on a
shared box — indicative of the direction and rough magnitude, not a controlled
benchmark. But the direction is not noise, and the mechanism is clear: the GPU
serializes the work regardless of how many requests hold a slot, so extra
concurrency adds scheduling overhead and multiplies peak activation memory
without buying any parallelism. On a Jetson sharing unified memory with other
workloads, that memory comes out of something else.

**The corollary matters for how you read your logs.** A `503` from this service
is the *acquire timeout*, not a capacity signal. It does not mean "buy a bigger
box" or "raise the limit". It means "requests are arriving faster than one GPU
can serve them", and the two correct responses are to retry with backoff
(notebook 07) and to send fewer, larger requests — which is what the rest of
this notebook is about.

## The four batch routes

`POST /extract_entities_batch`, `/classify_text_batch`,
`/extract_relations_batch`, `/extract_multitask_batch`.

All four:

- require a **boundary** checkpoint (`501` otherwise — same reason as relations
  in notebook 05);
- take `texts` (a list) instead of `text`;
- return `{"results": [...]}` **positionally aligned** with the `texts` you
  sent — result `i` is for text `i`, always;
- hold **one** inference slot for the whole batch, instead of contending for the
  semaphore once per document;
- accept every inference option from notebook 04, plus `batch_size`;
- are bounded by `MAX_BATCH_SIZE` **and** `MAX_BATCH_CHARS`.

That third and fourth point are the two sources of the speedup, and they are
worth separating.

### Why batching helps when the GPU is already saturated

This is the apparent paradox. If one document already saturates the GPU, how can
sending 32 at once be ten times faster per document?

Because "saturated" describes the *device during a forward pass*, not the
*wall-clock life of a request*. A single-document request spends time on:

1. HTTP parse, validation, schema construction — CPU, GPU idle;
2. **acquiring the inference semaphore** — pure waiting, GPU busy with someone
   else or idle;
3. tokenization and moving tensors to the device — GPU mostly idle;
4. the forward pass — GPU busy;
5. decoding predictions and serializing JSON — GPU idle.

Only step 4 uses the accelerator. Steps 1, 2, 3 and 5 are per-request overhead
that a batch pays **once** instead of 32 times. Step 2 in particular is
brutal on a one-slot box: 32 documents as 32 requests means 32 separate
acquisitions, each one a chance to queue behind something else.

There is a second, independent effect inside step 4. A GPU running a forward
pass on a single short document is not compute-bound at all — it is bound by
kernel launch overhead and memory bandwidth, with most of its arithmetic units
idle. Stacking 32 documents into one batched pass fills those units. The batched
forward pass takes nowhere near 32 times as long as the single one.

So batching wins on both axes at once: it amortizes the non-GPU overhead across
documents, and it uses the GPU more efficiently during the part that is on-GPU.
Neither of those requires any *additional* concurrency, which is exactly why it
works where thread pools failed.

In [6]:
BATCH_TEXTS = [
    "Apple CEO Tim Cook announced iPhone 15 in Cupertino.",
    "Satya Nadella leads Microsoft from Redmond.",
]

if not IS_BOUNDARY:
    print("skipping: the batch routes need a GLiNER2.5 boundary checkpoint")
else:
    print("--- extract_entities_batch ---")
    show(post("/extract_entities_batch",
              {"texts": BATCH_TEXTS, "labels": ["company", "person", "location"]}))

    print("\n--- classify_text_batch ---")
    show(post("/classify_text_batch", {
        "texts": ["The battery is terrible.", "Works flawlessly, love it."],
        "labels": {"sentiment": ["positive", "negative", "neutral"]},
    }))

    print("\n--- extract_relations_batch ---")
    show(post("/extract_relations_batch", {
        "texts": ["Satya Nadella, CEO of Microsoft, met Sam Altman of OpenAI.",
                  "Tim Cook runs Apple from Cupertino."],
        "relations": ["works_for", "met_with"],
    }))

    print("\n--- extract_multitask_batch ---")
    show(post("/extract_multitask_batch", {
        "texts": BATCH_TEXTS,
        "schema_config": {
            "entities": ["company", "person"],
            "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
        },
    }))

--- extract_entities_batch ---


{
  "results": [
    {
      "entities": {
        "company": [
          "Apple"
        ],
        "person": [
          "Tim Cook"
        ],
        "location": [
          "Cupertino"
        ]
      }
    },
    {
      "entities": {
        "company": [
          "Microsoft"
        ],
        "person": [
          "Satya Nadella"
        ],
        "location": [
          "Redmond"
        ]
      }
    }
  ]
}

--- classify_text_batch ---


{
  "results": [
    {
      "sentiment": "negative"
    },
    {
      "sentiment": "positive"
    }
  ]
}

--- extract_relations_batch ---


{
  "results": [
    {
      "relation_extraction": {
        "works_for": [
          [
            "Satya Nadella",
            "Microsoft"
          ]
        ],
        "met_with": [
          [
            "Satya Nadella",
            "Sam Altman"
          ]
        ]
      }
    },
    {
      "relation_extraction": {
        "works_for": [
          [
            "Tim Cook",
            "Cupertino"
          ]
        ],
        "met_with": []
      }
    }
  ]
}

--- extract_multitask_batch ---


{
  "results": [
    {
      "entities": {
        "company": [
          "Apple"
        ],
        "person": [
          "Tim Cook"
        ]
      },
      "sentiment": "positive"
    },
    {
      "entities": {
        "company": [
          "Microsoft"
        ],
        "person": [
          "Satya Nadella"
        ]
      },
      "sentiment": "positive"
    }
  ]
}


Look at the second `extract_relations_batch` result before moving on. On
`gliner2.5-base-v1` this text — *"Tim Cook runs Apple from Cupertino."* —
typically yields `works_for` as `["Tim Cook", "Cupertino"]`: a place, not the
company sitting in the same sentence.

That is notebook 05's "no type constraint" warning showing up again, and it is
worth re-noticing here because **batching multiplies it**. A validator you might
have skipped for a handful of interactive calls is not optional when you are
pushing thousands of documents through unattended. Batch the validation too.

### `include_spans` on a batch: offsets are per document

Offsets returned by a batch route index into **each individual document**, not
into some concatenation of the batch. That is the only sane choice — you sent
separate texts and you get separate coordinate spaces back — but it is worth
asserting rather than assuming, since an off-by-one document here would corrupt
every highlight in a UI.

In [7]:
if IS_BOUNDARY:
    spans_batch = post("/extract_entities_batch", {
        "texts": BATCH_TEXTS,
        "labels": ["company", "person"],
        "include_spans": True,
        "batch_size": 8,
    })
    show(spans_batch)

    for doc, result in zip(BATCH_TEXTS, spans_batch["results"]):
        for label, items in result["entities"].items():
            for item in items:
                assert doc[item["start"]:item["end"]] == item["text"], (doc, item)
    print("\nper-document offsets verified")
else:
    print("skipped: needs a boundary checkpoint")

{
  "results": [
    {
      "entities": {
        "company": [
          {
            "text": "Apple",
            "start": 0,
            "end": 5
          }
        ],
        "person": [
          {
            "text": "Tim Cook",
            "start": 10,
            "end": 18
          }
        ]
      }
    },
    {
      "entities": {
        "company": [
          {
            "text": "Microsoft",
            "start": 20,
            "end": 29
          }
        ],
        "person": [
          {
            "text": "Satya Nadella",
            "start": 0,
            "end": 13
          }
        ]
      }
    }
  ]
}

per-document offsets verified


## The two batch bounds

A batch is constrained by two independent limits:

- **`MAX_BATCH_SIZE`** — the number of documents (code default 64).
- **`MAX_BATCH_CHARS`** — the total characters summed across the batch.

Either overrun is a `413`.

**Why two?** Because a document count alone does not bound memory. Peak
activation memory during a batched forward pass scales with total tokens, not
with document count. Sixty-four documents of a few sentences each is trivial;
sixty-four documents of `MAX_TEXT_CHARS` (20 000) each is 1.28 million
characters, and that is what OOMed the box. The character bound is the one that
actually protects the GPU. The document bound is a cheap sanity cap on top of it.

**Both are environment variables, and a given deployment may run much tighter
than the code defaults.** So discover them from the live service rather than
hardcoding — a deliberate overrun reports the live value in the `413` detail,
which is the most reliable source there is.

In [8]:
import re

MAX_BATCH_SIZE = 64          # code defaults, replaced by discovery below
MAX_BATCH_CHARS = 200_000


def _limit_from(detail, key, fallback):
    m = re.search(rf"{key}=(\d+)", str(detail))
    return int(m.group(1)) if m else fallback


if IS_BOUNDARY:
    print("--- too many documents ---")
    status, body = post_raw(
        "/extract_entities_batch",
        {"texts": ["hello world"] * (MAX_BATCH_SIZE + 1), "labels": ["person"]})
    print("HTTP", status, "|", body)
    MAX_BATCH_SIZE = _limit_from(body.get("detail"), "MAX_BATCH_SIZE", MAX_BATCH_SIZE)

    print("\n--- too many characters (only 15 documents) ---")
    status, body = post_raw(
        "/extract_entities_batch",
        {"texts": ["a" * 19_000] * 15, "labels": ["person"]})
    print("HTTP", status, "|", body)
    MAX_BATCH_CHARS = _limit_from(body.get("detail"), "MAX_BATCH_CHARS", MAX_BATCH_CHARS)

    print(f"\nlive bounds on this deployment: "
          f"MAX_BATCH_SIZE={MAX_BATCH_SIZE} MAX_BATCH_CHARS={MAX_BATCH_CHARS}")
else:
    print("skipped: needs a boundary checkpoint")

--- too many documents ---
HTTP 413 | {'detail': "'texts' exceeds MAX_BATCH_SIZE=64."}

--- too many characters (only 15 documents) ---
HTTP 413 | {'detail': 'batch totals 285000 chars, exceeding MAX_BATCH_CHARS=40000. Split the batch.'}

live bounds on this deployment: MAX_BATCH_SIZE=64 MAX_BATCH_CHARS=40000


Note the second case: **15 documents, well inside a 64-document cap, and still
rejected** — on characters. That is the whole reason to chunk against both
bounds rather than just slicing your list into groups of 64.

If the discovered `MAX_BATCH_CHARS` above is smaller than the 200 000 code
default, that is this deployment being deliberately conservative about GPU
memory. Read the value; do not assume the default.

The chunker below respects both. It is worth having in your client library
rather than re-deriving each time.

In [9]:
def chunk_batches(texts, max_size=None, max_chars=None):
    """Split texts into batches respecting BOTH server bounds."""
    max_size = MAX_BATCH_SIZE if max_size is None else max_size
    max_chars = MAX_BATCH_CHARS if max_chars is None else max_chars
    batch, chars = [], 0
    for t in texts:
        if batch and (len(batch) >= max_size or chars + len(t) > max_chars):
            yield batch
            batch, chars = [], 0
        batch.append(t)
        chars += len(t)
    if batch:
        yield batch


sizes = [len(b) for b in chunk_batches(["a" * 19_000] * 40)]
print("40 x 19k-char documents chunk into batches of:", sizes)
print("(character-bound, not document-bound - note none of these reach MAX_BATCH_SIZE)")

40 x 19k-char documents chunk into batches of: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
(character-bound, not document-bound - note none of these reach MAX_BATCH_SIZE)


One edge case the chunker does not handle, deliberately: a single document
longer than `max_chars` will be emitted in a batch of one and rejected `413` by
the server. That is correct — it is a `MAX_TEXT_CHARS` problem, not a batching
problem, and silently truncating it would be worse. Split oversized documents
before they reach here.

## Measured: batched vs sequential

Now the comparison that decides how you write a bulk job. Two ways to get
entities, a classification and a structured record out of 32 documents:

- **Sequential** — three single-document calls per document. 96 HTTP requests,
  96 trips through the inference semaphore, 96 encodes of text.
- **Batched multi-task** — one `/extract_multitask_batch` call per chunk. One
  encode per document, one semaphore acquisition per chunk.

Reference numbers measured on `jarvita-agx` over 32 documents against
`fastino/gliner2.5-base-v1`:

| Approach | Per document | Documents/min |
|---|---|---|
| 3 sequential single calls per document | 274.9 ms | 218 |
| 1 `/extract_multitask_batch` call | 25.8 ms | 2326 |

**10.7x.** One run, shared box — indicative of the order of magnitude, not a
controlled benchmark.

**Expect your own run to differ, possibly by a lot.** The sequential path is the
noisy one: it makes 96 separate requests, so it absorbs every bit of contention
on the box, while the batched path makes one request and is largely insulated.
Two runs captured while `jarvita-agx` was busy showed 922.6 ms/document
sequential against 24.0 ms/document batched (**38.4x**), and 895.3 against 11.8
(**76.0x**) — not because batching got better between them, but because the
sequential baseline was being starved. The same code, minutes apart, produced
ratios differing by a factor of two. Read this number as "order of magnitude",
never as a figure to quote.

What is stable across every run is the direction and the reason for it.

The gain has **two independent sources**, and it is worth keeping them apart
because they apply in different situations:

1. **Batching** amortizes per-request and per-acquisition overhead across
   documents, and fills the GPU during the forward pass.
2. **Multi-task** does three tasks in one forward pass instead of three.

Calling `/extract_entities_batch`, `/classify_text_batch` and
`/extract_structured` separately buys you (1) but not (2). If your three tasks
genuinely need different texts or different thresholds you have to give up (2);
otherwise, take both.

In [10]:
N_DOCS = 32

CORPUS = [
    f"Apple CEO Tim Cook announced record revenue in Cupertino in quarter {i}."
    for i in range(N_DOCS)
]

ENTITY_LABELS = ["company", "person", "location"]
CLS_LABELS = {"sentiment": ["positive", "negative"]}
STRUCT_SCHEMA = {"announcement": ["who::str", "what::str"]}

MULTITASK_SCHEMA = {
    "entities": ENTITY_LABELS,
    "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
    "structure": {
        "name": "announcement",
        "fields": [{"name": "who", "dtype": "str"}, {"name": "what", "dtype": "str"}],
    },
}

post("/extract_entities", {"text": CORPUS[0], "labels": ENTITY_LABELS})
print(f"warm-up done | {N_DOCS} documents | model:", get("/health")["model_id"])

warm-up done | 32 documents | model: fastino/gliner2.5-base-v1


In [11]:
# --- Sequential: three single-document calls per document ---
t0 = time.perf_counter()
for doc in CORPUS:
    post("/extract_entities", {"text": doc, "labels": ENTITY_LABELS})
    post("/classify_text", {"text": doc, "labels": CLS_LABELS})
    post("/extract_structured", {"text": doc, "schema": STRUCT_SCHEMA})
seq_ms = (time.perf_counter() - t0) * 1000

print(f"sequential (3 calls/doc) : {seq_ms:9.1f} ms total   "
      f"{seq_ms / N_DOCS:7.1f} ms/doc   {60_000 / (seq_ms / N_DOCS):7.0f} docs/min")

sequential (3 calls/doc) :   28649.8 ms total     895.3 ms/doc        67 docs/min


In [12]:
# --- Batched: one multitask call per chunk, chunked against both bounds ---
if not IS_BOUNDARY:
    print("skipped: /extract_multitask_batch needs a boundary checkpoint")
else:
    batches = list(chunk_batches(CORPUS))
    t0 = time.perf_counter()
    batched_results = []
    for batch in batches:
        r = post("/extract_multitask_batch",
                 {"texts": batch, "schema_config": MULTITASK_SCHEMA})
        batched_results.extend(r["results"])
    batch_ms = (time.perf_counter() - t0) * 1000

    assert len(batched_results) == N_DOCS, "results must align 1:1 with texts"

    print(f"multitask_batch          : {batch_ms:9.1f} ms total   "
          f"{batch_ms / N_DOCS:7.1f} ms/doc   {60_000 / (batch_ms / N_DOCS):7.0f} docs/min")
    print(f"\nspeedup: {seq_ms / batch_ms:.1f}x  "
          f"({len(batches)} HTTP request(s) instead of {N_DOCS * 3})")
    print("\nShared box, one run - indicative, not a controlled benchmark.")
    print("\nfirst result:")
    show(batched_results[0])

multitask_batch          :     377.2 ms total      11.8 ms/doc      5090 docs/min

speedup: 76.0x  (1 HTTP request(s) instead of 96)

Shared box, one run - indicative, not a controlled benchmark.

first result:
{
  "announcement": [
    {
      "who": "Tim Cook",
      "what": "record revenue"
    }
  ],
  "entities": {
    "company": [
      "Apple"
    ],
    "person": [
      "Tim Cook"
    ],
    "location": [
      "Cupertino"
    ]
  },
  "sentiment": "positive"
}


Whatever multiple you got, note the *other* number in that output: one HTTP
request instead of 96. Even setting the model aside entirely, that is 95 fewer
opportunities to hit a `503`, 95 fewer retries to write logic for, and 95 fewer
round trips of network latency. Batching simplifies the failure model as much as
it speeds things up.

## Two dead ends, measured

Both of these are plausible-sounding optimizations that were tried on this
deployment and did not work. They are documented so you do not spend a week
rediscovering them.

**Schema caching between batch calls.** The idea: the schema is identical across
calls in a bulk job, so cache the constructed schema object and skip rebuilding
it. Measured over 16 documents: **127.41 ms with, 127.42 ms without.** A 0.01 ms
difference — noise, not a signal. Schema construction is simply not on the
critical path next to a GPU forward pass. Do not build this.

**Raising `MAX_CONCURRENT_INFERENCES`.** Covered at the top of this notebook:
8.0 docs/s at one slot, 3.4 docs/s at eight. It made things worse. The GPU
serializes the work regardless, and extra concurrency multiplies peak activation
memory — on a box sharing unified memory with an LLM, that memory is taken from
something else.

A third thing to be careful with rather than a flat dead end: **bigger batches
without watching memory.** `MAX_BATCH_CHARS` is the bound that actually holds
back an OOM, so raising it is the OOM lever. If you do raise it, watch
`docker stats` through a full-size batch before trusting the new value.

## Try this yourself

Separate the two sources of the speedup. Run the same 32 documents through
**three separate batch calls** — `/extract_entities_batch`,
`/classify_text_batch`, `/extract_structured` per document — and compare that
against the single multi-task batch.

Predict first: is three-batch-calls closer to the sequential number or to the
multi-task-batch number?

In [13]:
if IS_BOUNDARY:
    batches = list(chunk_batches(CORPUS))

    t0 = time.perf_counter()
    for batch in batches:
        post("/extract_entities_batch", {"texts": batch, "labels": ENTITY_LABELS})
        post("/classify_text_batch", {"texts": batch, "labels": CLS_LABELS})
        # No structured batch route exists, so structure stays per document.
        for doc in batch:
            post("/extract_structured", {"text": doc, "schema": STRUCT_SCHEMA})
    three_batch_ms = (time.perf_counter() - t0) * 1000

    print(f"3 separate batch paths : {three_batch_ms:9.1f} ms total   "
          f"{three_batch_ms / N_DOCS:7.1f} ms/doc")
    print(f"\nvs sequential      : {seq_ms / three_batch_ms:5.1f}x faster")
    print(f"vs multitask_batch : {three_batch_ms / batch_ms:5.1f}x slower")
else:
    print("skipped: needs a boundary checkpoint")

3 separate batch paths :    8547.6 ms total     267.1 ms/doc

vs sequential      :   3.4x faster
vs multitask_batch :  22.7x slower


**Discussion.** Three separate batch paths land in between, and the reason is
visible in the code: the entity and classification work batches cleanly, but
there is **no `/extract_structured_batch` route**, so the structured task falls
back to one request per document. That single un-batchable task drags the whole
run back toward the sequential number.

This is the practical lesson, and it generalizes past this API: **the slowest
un-batched step sets your floor.** Optimizing two of three stages while the
third stays per-document buys you a fraction of the available win.

It is also the strongest argument for `/extract_multitask_batch` specifically.
It is not merely a convenience wrapper over the other three — it is the only
route that batches structured extraction at all, because structure only exists
in batch form as part of a multi-task schema. If your pipeline needs entities,
classification and records over a corpus, multi-task batch is not one option
among several. It is the option.

One caveat on the numbers: this corpus is 32 near-identical short sentences,
which is close to the best case for batching. Real corpora with varied document
lengths batch less evenly, because a batch is bounded by its total characters
and a single long document forces a smaller batch around it. Expect a smaller
multiple on real data — and measure it on yours rather than trusting this cell.

## What you learned

- The GPU is already saturated by one request. Naive concurrency does not add
  parallelism, it adds queueing — and past the acquire timeout it converts work
  into `503`s.
- Raising `MAX_CONCURRENT_INFERENCES` measured **worse**: 8.0 docs/s at one
  slot, 3.4 docs/s at eight. A `503` is the acquire timeout, not a capacity
  signal.
- Batching wins on two axes without adding concurrency: it amortizes
  per-request and per-acquisition overhead, and it fills a GPU that a single
  short document leaves mostly idle.
- The four batch routes need a boundary checkpoint, return `results` positionally
  aligned with `texts`, hold one inference slot for the whole batch, and take
  every inference option plus `batch_size`.
- Two bounds apply — `MAX_BATCH_SIZE` (documents) and `MAX_BATCH_CHARS`
  (characters summed). The character bound is the one protecting the GPU.
  Discover the live values from the `413` detail; chunk against both.
- Measured 274.9 → 25.8 ms/document (10.7x over 32 documents, one run, shared
  box) for one multi-task batch call versus three sequential single calls. Two
  independent sources: batching, and multi-task.
- Dead ends: schema caching (127.41 vs 127.42 ms — noise) and more concurrency.
  Do not build either.
- The slowest un-batched step sets your floor, which is why
  `/extract_multitask_batch` matters — it is the only way to batch structured
  extraction.

## Next

**[07 — Operating it in production](07-operating-in-production.ipynb)** — the
full error taxonomy, retry with backoff, monitoring, and a deployment checklist.